# 08. Attention core — DeepSeek-V4-Flash full attention schedule at reduced tensor width

This notebook preserves the **released DeepSeek-V4-Flash attention topology** and reduces only tensor widths / batch / sequence length.

Preserved structural constants from the released config:

- 43 decoder attention sites
- 64 query heads and 1 shared KV head
- sliding window = 128
- CSA compression ratio = 4
- HCA compression ratio = 128
- Lightning Indexer: 64 heads, top-k = 512
- grouped output projection with 8 groups
- layer compression schedule exactly equal to the released 43-entry `compress_ratios`

Reduced: hidden width, per-head width, low-rank widths, batch size and test sequence length.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))
print("device:", device)

In [ ]:
V4_FLASH_ALL_COMPRESS_RATIOS = [0, 0]
for _ in range(20):
    V4_FLASH_ALL_COMPRESS_RATIOS.extend([4, 128])
V4_FLASH_ALL_COMPRESS_RATIOS.extend([4, 0])

# The released list contains 43 main-model entries plus one MTP entry.
assert len(V4_FLASH_ALL_COMPRESS_RATIOS) == 44
V4_FLASH_COMPRESS_RATIOS = V4_FLASH_ALL_COMPRESS_RATIOS[:-1]
V4_FLASH_MTP_COMPRESS_RATIO = V4_FLASH_ALL_COMPRESS_RATIOS[-1]

assert len(V4_FLASH_COMPRESS_RATIOS) == 43
assert V4_FLASH_COMPRESS_RATIOS[:4] == [0, 0, 4, 128]
assert V4_FLASH_COMPRESS_RATIOS[-1] == 4
assert V4_FLASH_MTP_COMPRESS_RATIO == 0

V4_QUERY_HEADS = 64
V4_KV_HEADS = 1
V4_SLIDING_WINDOW = 128
V4_INDEX_HEADS = 64
V4_INDEX_TOPK = 512
V4_OUTPUT_GROUPS = 8

print("V4 attention sites:", len(V4_FLASH_COMPRESS_RATIOS))
print("CSA sites:", V4_FLASH_COMPRESS_RATIOS.count(4))
print("HCA sites:", V4_FLASH_COMPRESS_RATIOS.count(128))
print("sliding-only sites:", V4_FLASH_COMPRESS_RATIOS.count(0))

## 1. Partial RoPE and grouped output projection

V4 rotates only a subspace of Q/K. After attention, the RoPE part of the head output receives the conjugate rotation before the grouped low-rank output projection.

In [ ]:
def rms_without_weight(x, eps=1e-6):
    x_float = x.float()
    scale = torch.rsqrt(
        x_float.square().mean(dim=-1, keepdim=True) + eps
    )
    return (x_float * scale).to(x.dtype)


def rope_cos_sin(position_ids, rope_dim, base=10000.0):
    pair_index = torch.arange(
        0,
        rope_dim,
        2,
        device=position_ids.device,
        dtype=torch.float32,
    )
    inverse_frequency = 1.0 / (base ** (pair_index / rope_dim))
    angles = position_ids.float()[:, None] * inverse_frequency[None]
    return angles.cos(), angles.sin()


def rotate_rope_subspace(x, position_ids, rope_dim, conjugate=False):
    if rope_dim == 0:
        return x

    content = x[..., :-rope_dim]
    rope = x[..., -rope_dim:]
    cosine, sine = rope_cos_sin(position_ids, rope_dim)

    if rope.ndim == 4:
        # [batch, sequence, heads, rope_dim]
        cosine = cosine[None, :, None, :]
        sine = sine[None, :, None, :]
    elif rope.ndim == 3:
        # [batch, sequence, rope_dim]
        cosine = cosine[None, :, :]
        sine = sine[None, :, :]
    else:
        raise ValueError(
            f"unsupported RoPE tensor shape: {tuple(rope.shape)}"
        )

    even = rope[..., 0::2]
    odd = rope[..., 1::2]

    if conjugate:
        sine = -sine

    rotated_even = even * cosine - odd * sine
    rotated_odd = even * sine + odd * cosine
    rotated = torch.stack(
        [rotated_even, rotated_odd],
        dim=-1,
    ).flatten(-2)
    return torch.cat([content, rotated], dim=-1)


class GroupedOutputProjection(nn.Module):
    def __init__(
        self,
        query_heads=64,
        head_dim=4,
        groups=8,
        low_rank=16,
        model_dim=32,
    ):
        super().__init__()

        total_width = query_heads * head_dim
        assert total_width % groups == 0

        self.groups = groups
        self.group_width = total_width // groups
        self.group_down = nn.ModuleList(
            [
                nn.Linear(self.group_width, low_rank, bias=False)
                for _ in range(groups)
            ]
        )
        self.up = nn.Linear(
            groups * low_rank,
            model_dim,
            bias=False,
        )

    def forward(self, heads):
        flattened = heads.flatten(2)
        chunks = flattened.split(self.group_width, dim=-1)
        low_rank_chunks = [
            projection(chunk)
            for projection, chunk in zip(self.group_down, chunks)
        ]
        return self.up(torch.cat(low_rank_chunks, dim=-1))

## 2. Exact compressor topologies

CSA constructs each compressed entry from `Ca` of the previous 4-token block and `Cb` of the current block. HCA compresses every 128-token block directly. Both use learned element-wise gates and RMS normalization.

In [ ]:
class CSACompressor(nn.Module):
    def __init__(self, model_dim=32, head_dim=4, rate=4):
        super().__init__()
        assert rate == 4

        self.rate = rate
        self.head_dim = head_dim
        self.kv_projection = nn.Linear(
            model_dim,
            2 * head_dim,
            bias=False,
        )
        self.gate_projection = nn.Linear(
            model_dim,
            2 * head_dim,
            bias=False,
        )
        self.position_bias = nn.Parameter(
            torch.zeros(rate, 2 * head_dim)
        )
        self.norm = nn.RMSNorm(head_dim)

    def forward(self, hidden):
        batch_size, sequence_length, _ = hidden.shape
        usable_length = (sequence_length // self.rate) * self.rate
        hidden = hidden[:, :usable_length]

        if usable_length == 0:
            empty = hidden.new_zeros(batch_size, 0, self.head_dim)
            positions = torch.empty(
                0,
                dtype=torch.long,
                device=hidden.device,
            )
            return empty, positions

        block_count = usable_length // self.rate
        kv = self.kv_projection(hidden).view(
            batch_size,
            block_count,
            self.rate,
            2 * self.head_dim,
        )
        gate = self.gate_projection(hidden).view_as(kv)
        gate = gate + self.position_bias

        ca, cb = kv.chunk(2, dim=-1)
        gate_a, gate_b = gate.chunk(2, dim=-1)

        entries = hidden.new_zeros(
            batch_size,
            block_count,
            2 * self.rate,
            self.head_dim,
        )
        logits = hidden.new_full(
            entries.shape,
            float("-inf"),
        )

        entries[:, :, self.rate:] = cb
        logits[:, :, self.rate:] = gate_b

        if block_count > 1:
            entries[:, 1:, :self.rate] = ca[:, :-1]
            logits[:, 1:, :self.rate] = gate_a[:, :-1]

        weights = logits.softmax(dim=2, dtype=torch.float32)
        weights = weights.to(entries.dtype)
        compressed = self.norm((entries * weights).sum(dim=2))

        positions = (
            torch.arange(block_count, device=hidden.device)
            * self.rate
        )
        return compressed, positions


class HCACompressor(nn.Module):
    def __init__(self, model_dim=32, head_dim=4, rate=128):
        super().__init__()
        assert rate == 128

        self.rate = rate
        self.head_dim = head_dim
        self.kv_projection = nn.Linear(
            model_dim,
            head_dim,
            bias=False,
        )
        self.gate_projection = nn.Linear(
            model_dim,
            head_dim,
            bias=False,
        )
        self.position_bias = nn.Parameter(
            torch.zeros(rate, head_dim)
        )
        self.norm = nn.RMSNorm(head_dim)

    def forward(self, hidden):
        batch_size, sequence_length, _ = hidden.shape
        usable_length = (sequence_length // self.rate) * self.rate
        hidden = hidden[:, :usable_length]

        if usable_length == 0:
            empty = hidden.new_zeros(batch_size, 0, self.head_dim)
            positions = torch.empty(
                0,
                dtype=torch.long,
                device=hidden.device,
            )
            return empty, positions

        block_count = usable_length // self.rate
        kv = self.kv_projection(hidden).view(
            batch_size,
            block_count,
            self.rate,
            self.head_dim,
        )
        gate = self.gate_projection(hidden).view_as(kv)
        gate = gate + self.position_bias

        weights = gate.softmax(dim=2, dtype=torch.float32)
        weights = weights.to(kv.dtype)
        compressed = self.norm((kv * weights).sum(dim=2))

        positions = (
            torch.arange(block_count, device=hidden.device)
            * self.rate
        )
        return compressed, positions

## 3. Lightning Indexer

The indexer keeps the released 64 index heads and top-k 512. When a short test sequence contains fewer than 512 compressed entries, `min(top_k, available_entries)` is only a runtime bound, not a changed architectural hyperparameter.

In [ ]:
class LightningIndexer(nn.Module):
    def __init__(
        self,
        model_dim=32,
        q_rank=8,
        index_heads=64,
        index_head_dim=2,
        top_k=512,
        rate=4,
        rope_dim=2,
    ):
        super().__init__()

        assert index_heads == 64
        assert top_k == 512
        assert rate == 4

        self.index_heads = index_heads
        self.index_head_dim = index_head_dim
        self.top_k = top_k
        self.rate = rate
        self.rope_dim = rope_dim

        self.compressor = CSACompressor(
            model_dim=model_dim,
            head_dim=index_head_dim,
            rate=rate,
        )
        self.query_projection = nn.Linear(
            q_rank,
            index_heads * index_head_dim,
            bias=False,
        )
        self.head_weight = nn.Linear(
            model_dim,
            index_heads,
            bias=False,
        )

    def forward(self, hidden, q_latent):
        batch_size, sequence_length, _ = hidden.shape
        token_positions = torch.arange(
            sequence_length,
            device=hidden.device,
        )

        keys, key_positions = self.compressor(hidden)
        if keys.size(1) == 0:
            empty = torch.empty(
                batch_size,
                sequence_length,
                0,
                dtype=torch.long,
                device=hidden.device,
            )
            return empty

        keys = rotate_rope_subspace(
            keys[:, :, None, :],
            key_positions,
            self.rope_dim,
        ).squeeze(2)

        queries = self.query_projection(q_latent).view(
            batch_size,
            sequence_length,
            self.index_heads,
            self.index_head_dim,
        )
        queries = rotate_rope_subspace(
            queries,
            token_positions,
            self.rope_dim,
        )

        per_head_score = torch.einsum(
            "bthd,bnd->bthn",
            queries.float(),
            keys.float(),
        )
        per_head_score = F.relu(per_head_score)
        per_head_score = per_head_score / math.sqrt(self.index_head_dim)

        head_weight = self.head_weight(hidden).float()
        head_weight = head_weight / math.sqrt(self.index_heads)
        score = (
            per_head_score
            * head_weight[..., None]
        ).sum(dim=2)

        causal_entry_count = (
            token_positions[None] + 1
        ) // self.rate
        entry_ids = torch.arange(
            keys.size(1),
            device=hidden.device,
        )
        valid = entry_ids[None, None] < causal_entry_count[..., None]
        score = score.masked_fill(~valid, float("-inf"))

        runtime_top_k = min(self.top_k, keys.size(1))
        selected = score.topk(runtime_top_k, dim=-1).indices
        selected_valid = selected < causal_entry_count[..., None]
        selected = torch.where(
            selected_valid,
            selected,
            -torch.ones_like(selected),
        )
        return selected

## 4. V4 shared-KV attention

Each layer always has the local shared-K=V MQA path. CSA adds only Lightning-Indexer-selected compressed entries; HCA adds all causally available 128× compressed entries. Attention sinks are appended to the logits but not to the value sum.

In [ ]:
class DeepSeekV4Attention(nn.Module):
    def __init__(
        self,
        compression_ratio,
        model_dim=32,
        query_heads=64,
        head_dim=4,
        rope_dim=2,
        q_rank=8,
        output_groups=8,
        output_rank=16,
        sliding_window=128,
    ):
        super().__init__()

        assert query_heads == 64
        assert output_groups == 8
        assert sliding_window == 128
        assert compression_ratio in {0, 4, 128}

        self.compression_ratio = compression_ratio
        self.query_heads = query_heads
        self.head_dim = head_dim
        self.rope_dim = rope_dim
        self.sliding_window = sliding_window

        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_down_norm = nn.RMSNorm(q_rank)
        self.q_up = nn.Linear(
            q_rank,
            query_heads * head_dim,
            bias=False,
        )

        self.shared_kv_projection = nn.Linear(
            model_dim,
            head_dim,
            bias=False,
        )
        self.shared_kv_norm = nn.RMSNorm(head_dim)
        self.attention_sink = nn.Parameter(
            torch.zeros(query_heads)
        )

        self.output_projection = GroupedOutputProjection(
            query_heads=query_heads,
            head_dim=head_dim,
            groups=output_groups,
            low_rank=output_rank,
            model_dim=model_dim,
        )

        if compression_ratio == 4:
            self.compressor = CSACompressor(
                model_dim,
                head_dim,
                rate=4,
            )
            self.indexer = LightningIndexer(
                model_dim=model_dim,
                q_rank=q_rank,
                index_heads=64,
                index_head_dim=2,
                top_k=512,
                rate=4,
                rope_dim=2,
            )
        elif compression_ratio == 128:
            self.compressor = HCACompressor(
                model_dim,
                head_dim,
                rate=128,
            )
            self.indexer = None
        else:
            self.compressor = None
            self.indexer = None

    def _queries(self, hidden, positions):
        q_latent = self.q_down_norm(self.q_down(hidden))
        query = self.q_up(q_latent).view(
            hidden.size(0),
            hidden.size(1),
            self.query_heads,
            self.head_dim,
        )
        query = rms_without_weight(query)
        query = rotate_rope_subspace(
            query,
            positions,
            self.rope_dim,
        )
        return query, q_latent

    def _local_shared_kv(self, hidden, positions):
        kv = self.shared_kv_norm(
            self.shared_kv_projection(hidden)
        )
        kv = rotate_rope_subspace(
            kv[:, :, None, :],
            positions,
            self.rope_dim,
        ).squeeze(2)
        return kv

    def _attend_one_token(self, query, kv, valid):
        score = torch.einsum(
            "bhd,bkd->bhk",
            query,
            kv,
        ) / math.sqrt(self.head_dim)
        score = score.masked_fill(
            ~valid[:, None],
            torch.finfo(score.dtype).min,
        )

        sink = self.attention_sink[None, :, None].expand(
            query.size(0),
            -1,
            1,
        )
        logits = torch.cat([score, sink], dim=-1)
        weights = logits.softmax(dim=-1)[..., :-1]
        return torch.einsum(
            "bhk,bkd->bhd",
            weights,
            kv,
        )

    def forward(self, hidden):
        batch_size, sequence_length, _ = hidden.shape
        positions = torch.arange(sequence_length, device=hidden.device)

        query, q_latent = self._queries(hidden, positions)
        local_kv = self._local_shared_kv(hidden, positions)

        compressed = None
        compressed_positions = None
        selected = None
        if self.compressor is not None:
            compressed, compressed_positions = self.compressor(hidden)
            compressed = rotate_rope_subspace(
                compressed[:, :, None, :],
                compressed_positions,
                self.rope_dim,
            ).squeeze(2)

        if self.indexer is not None:
            selected = self.indexer(hidden, q_latent)

        outputs = []
        batch_ids = torch.arange(batch_size, device=hidden.device)[:, None]

        for token_index in range(sequence_length):
            local_start = max(
                0,
                token_index - self.sliding_window + 1,
            )
            kv_parts = [
                local_kv[:, local_start : token_index + 1]
            ]
            valid_parts = [
                torch.ones(
                    batch_size,
                    token_index - local_start + 1,
                    dtype=torch.bool,
                    device=hidden.device,
                )
            ]

            if self.compression_ratio == 4 and selected.size(-1) > 0:
                ids = selected[:, token_index]
                valid = ids >= 0
                safe_ids = ids.clamp_min(0)
                kv_parts.append(compressed[batch_ids, safe_ids])
                valid_parts.append(valid)

            if self.compression_ratio == 128 and compressed.size(1) > 0:
                causal_count = (token_index + 1) // 128
                kv_parts.append(compressed)
                compressed_ids = torch.arange(
                    compressed.size(1),
                    device=hidden.device,
                )
                valid_parts.append(
                    (compressed_ids[None] < causal_count).expand(
                        batch_size,
                        -1,
                    )
                )

            kv = torch.cat(kv_parts, dim=1)
            valid = torch.cat(valid_parts, dim=1)
            outputs.append(
                self._attend_one_token(
                    query[:, token_index],
                    kv,
                    valid,
                )
            )

        heads = torch.stack(outputs, dim=1)
        heads = rotate_rope_subspace(
            heads,
            positions,
            self.rope_dim,
            conjugate=True,
        )
        return self.output_projection(heads)

## 5. Full released 43-layer attention schedule

The module list itself has all 43 sites. Widths are small, but the schedule is not shortened.

In [ ]:
class DeepSeekV4FlashAttentionStack(nn.Module):
    def __init__(self, model_dim=32):
        super().__init__()

        self.compress_ratios = list(V4_FLASH_COMPRESS_RATIOS)
        self.layers = nn.ModuleList(
            [
                DeepSeekV4Attention(
                    compression_ratio=ratio,
                    model_dim=model_dim,
                )
                for ratio in self.compress_ratios
            ]
        )

    def forward(self, hidden):
        for attention in self.layers:
            hidden = hidden + attention(hidden)
        return hidden


model = DeepSeekV4FlashAttentionStack().to(device)

assert len(model.layers) == 43
assert model.compress_ratios == V4_FLASH_COMPRESS_RATIOS
assert all(layer.query_heads == 64 for layer in model.layers)
assert all(layer.sliding_window == 128 for layer in model.layers)
assert all(layer.output_projection.groups == 8 for layer in model.layers)

csa_layers = [layer for layer in model.layers if layer.compression_ratio == 4]
hca_layers = [layer for layer in model.layers if layer.compression_ratio == 128]

assert all(layer.indexer.top_k == 512 for layer in csa_layers)
assert all(layer.indexer.index_heads == 64 for layer in csa_layers)
assert all(layer.compressor.rate == 4 for layer in csa_layers)
assert all(layer.compressor.rate == 128 for layer in hca_layers)

# Small end-to-end differentiability check.
hidden = torch.randn(1, 4, 32, device=device)
output = model(hidden)
loss = output.square().mean()
loss.backward()

print("full schedule output:", output.shape)
print("layers:", len(model.layers))
print("CSA/HCA/sliding:", len(csa_layers), len(hca_layers), model.compress_ratios.count(0))

## 6. Exercise the 128× HCA compressor separately

The full-stack sanity input is intentionally short. This separate check ensures that the real 128-token HCA grouping path is executable rather than dead code.

In [ ]:
hca_check = HCACompressor(
    model_dim=8,
    head_dim=2,
    rate=128,
).to(device)

hca_input = torch.randn(1, 128, 8, device=device)
hca_output, hca_positions = hca_check(hca_input)

assert hca_output.shape == (1, 1, 2)
assert hca_positions.tolist() == [0]

print("HCA 128x output:", hca_output.shape)
print("HCA positions:", hca_positions.tolist())

## Structural checklist

The assertions above check every non-width architectural constant demonstrated here: 43 sites, exact `compress_ratios`, 64 query heads, one shared KV path, 128-token local window, 4× CSA, 128× HCA, 64-head / top-512 Lightning Indexer, and 8-way grouped output projection.

References: released `DeepSeek-V4-Flash-Base` config and the public Transformers DeepSeek-V4 implementation.